In [19]:
import os, sys, re, csv, time, unicodedata
from pathlib import Path
from urllib.parse import urlparse, urljoin, unquote
from bs4 import BeautifulSoup
from bs4.element import NavigableString, Tag
# ------------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------------
project_root = Path.cwd().parent.parent   # adjust ../.. as needed
sys.path.append(str(project_root))
import config  # expects BASE_URL, FANDOM_DATA_DIR, LINKS_FILE
# ------------------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------------------
BASE_URL = config.BASE_URL.rstrip("/")
domain_full = urlparse(BASE_URL).netloc
domain = domain_full.split(".")[0]
FANDOM_DATA_DIR = Path(config.FANDOM_DATA_DIR)
LINKS_FILE = Path(config.LINKS_FILE)
if not LINKS_FILE.is_absolute():
    LINKS_FILE = FANDOM_DATA_DIR / LINKS_FILE.name
HTML_DIR   = FANDOM_DATA_DIR / f"{domain}_fandom_html"
SPANS_DIR  = FANDOM_DATA_DIR / f"{domain}_fandom_spans"
HTML_DIR.mkdir(parents=True, exist_ok=True)
SPANS_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = FANDOM_DATA_DIR / f"master_spans_{domain}.csv"


In [20]:
def extract_paragraph_and_spans(block: Tag, base_url: str):
    paragraph_text = ""
    spans = []

    def normalize_text(text: str) -> str:
        text = unicodedata.normalize("NFKC", text or "")
        return re.sub(r"\s+", " ", text)

    def needs_space(lc: str, rc: str) -> bool:
        def is_word_char(ch):
            return ch.isalnum() or ch in "_-’'°"
        return is_word_char(lc) and is_word_char(rc)

    def append_with_spacing(buf: str, chunk: str) -> (str, int):
        c = normalize_text(chunk)
        if not c:
            return buf, 0
        if buf.endswith(" ") and c.startswith(" "):
            c = c.lstrip()
            if not c:
                return buf, 0
        if buf and not buf.endswith(" ") and not c.startswith(" ") and needs_space(buf[-1], c[0]):
            c = " " + c
        return buf + c, len(c)

    for node in block.descendants:
        if isinstance(node, NavigableString):
            if getattr(node.parent, "name", None) != "a":
                paragraph_text, _ = append_with_spacing(paragraph_text, str(node))
        elif isinstance(node, Tag):
            if node.name == "br":
                paragraph_text, _ = append_with_spacing(paragraph_text, "\n")
            elif node.name == "a":
                link_text = normalize_text(node.get_text())
                if not link_text:
                    continue
                insert_space = (
                    paragraph_text and
                    not paragraph_text.endswith(" ") and
                    not link_text.startswith(" ") and
                    needs_space(paragraph_text[-1], link_text[0])
                )
                start = len(paragraph_text) + (1 if insert_space else 0)
                if insert_space:
                    paragraph_text += " "
                paragraph_text += link_text
                end = start + len(link_text)
                href = node.get("href", "")
                abs_url = href if href.startswith("http") else urljoin(base_url, href)
                spans.append({
                    "start": start,
                    "end": end,
                    "link_text": link_text,
                    "href": abs_url,
                })

    return paragraph_text, spans


In [21]:
def extract_spans_from_html(html_path: Path, base_url: str):
    html_content = html_path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html_content, "html.parser")

    # A helper to extract article_id and title (can be adapted)
    def extract_article_info(soup):
        aid, title = "", ""
        tag = soup.find(attrs={"wgArticleId": True}) or soup.find(attrs={"wgArticleID": True})
        if tag:
            v = tag.get("wgArticleId") or tag.get("wgArticleID")
            if v and str(v).isdigit():
                aid = int(v)
        if not aid:
            for sc in soup.find_all("script"):
                txt = sc.string or sc.get_text() or ""
                m = re.search(r'"wgArticleId"\s*:\s*(\d+)', txt)
                if m:
                    aid = int(m.group(1))
                    break
        for sc in soup.find_all("script"):
            txt = sc.string or sc.get_text() or ""
            m = re.search(r'"wgPageName"\s*:\s*"([^"]+)"', txt) or re.search(r'"wgTitle"\s*:\s*"([^"]+)"', txt)
            if m:
                title = m.group(1)
                break
        if not title:
            t = soup.find("title")
            if t:
                title = t.get_text(strip=True)
        return aid, title

    article_id, title = extract_article_info(soup)

    content_root = soup.select_one(".mw-parser-output") or soup.body or soup

    BLOCKS = "p, li, h1, h2, h3, h4, h5, h6, td, th, figcaption"

    rows = []
    paragraph_id = 0

    def normalize_text(text):
        return unicodedata.normalize("NFKC", text or "").strip()

    for block in content_root.select(BLOCKS):
        paragraph_text, spans = extract_paragraph_and_spans(block, base_url)
        if not paragraph_text.strip():
            continue
        paragraph_id += 1
        anchor_index = -1
        for s in spans:
            href = s["href"]
            if not (href.startswith("/wiki/") or href.startswith(base_url + "/wiki/")):
                continue
            anchor_index += 1
            abs_url = href if href.startswith("http") else urljoin(base_url, href.lstrip("/"))
            link_type = "self" if abs_url.rstrip("/") == base_url.rstrip("/") else "internal"
            rows.append({
                "article_id": article_id,
                "title": title,
                "paragraph_id": paragraph_id,
                "paragraph_text": paragraph_text,
                "anchor_ix": anchor_index,
                "link_text": s["link_text"],
                "start": s["start"],
                "end": s["end"],
                "link_type": link_type,
                "resolved_url": abs_url,
                "page_url": base_url,
            })

    return rows


In [23]:
html_path = Path("/home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/alldimensions_fandom_html/Voidonion.html")
print(f"File exists? {html_path.exists()}")
print(html_path.read_text()[:500])  # print first 500 characters


File exists? True
<!DOCTYPE html>
<html class="client-nojs sse-dplat-1692-rejected sse-other l2u-other" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Voidonion | All dimensions Wiki | Fandom</title>
<script>document.documentElement.className="client-js sse-dplat-1692-rejected sse-other l2u-other";RLCONF={"wgBreakFrames":false,"wgSeparatorTransformTable":["",""],"wgDigitTransformTable":["",""],"wgDefaultDateFormat":"dmy","wgMonthNames":["","January","February","March","April","May","June","July","Augus


In [24]:
soup = BeautifulSoup(html_path.read_text(encoding="utf-8", errors="replace"), "html.parser")
content_root = soup.select_one(".mw-parser-output") or soup.body or soup
print(content_root[:500] if content_root else "No content_root found")


KeyError: slice(None, 500, None)

In [22]:
# Path to your sample HTML local file
sample_html_path = Path("/home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/alldimensions_fandom_html/Voidonion.html")
BASE_URL = config.BASE_URL.rstrip("/")
# Extract spans
extracted_spans = extract_spans_from_html(sample_html_path, BASE_URL)

# Display some extracted spans to verify
for i, span in enumerate(extracted_spans[:10]):  # show first 10
    print(f"Span {i+1}:")
    print(f" Article ID: {span['article_id']}")
    print(f" Title: {span['title']}")
    print(f" Paragraph ID: {span['paragraph_id']}")
    print(f" Anchor Index: {span['anchor_ix']}")
    print(f" Link Text: {span['link_text']}")
    print(f" Start-End: {span['start']}-{span['end']}")
    print(f" Link Type: {span['link_type']}")
    print(f" Resolved URL: {span['resolved_url']}")
    print(f" Paragraph Text (excerpt): {span['paragraph_text'][:60]}...")
    print()
